# 🌾 Обучение модели сегментации полей в Google Colab

Этот notebook позволяет обучить модель сегментации сельскохозяйственных полей на GPU в Google Colab.

## Что делает этот notebook:
1. Подключает Google Drive
2. Устанавливает зависимости
3. Загружает данные
4. Обучает модель с GPU
5. Сохраняет результаты

## Перед запуском:
1. **Runtime → Change runtime type → GPU (T4 или лучше)**
2. Загрузите ваши данные на Google Drive в папку `MyDrive/agricultural_segmentation/`

## 1. Подключение Google Drive и настройка

In [1]:
from google.colab import drive
import os

# Подключить Google Drive
drive.mount('/content/drive')

# Создать рабочую директорию
WORK_DIR = '/content/agricultural_segmentation'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"Рабочая директория: {WORK_DIR}")

Mounted at /content/drive
Рабочая директория: /content/agricultural_segmentation


## 2. Установка зависимостей

In [2]:
%%capture
# Установить все необходимые библиотеки
!pip install albumentations==1.3.1
!pip install tensorboard
!pip install segmentation-models-pytorch  # Опционально для transfer learning

## 3. Проверка GPU

In [3]:
import torch

# Проверить доступность GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ GPU недоступен! Перейдите в Runtime → Change runtime type → GPU")

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
GPU Memory: 39.56 GB


## 4. Загрузка данных

### Вариант A: Загрузить из Google Drive (если уже загружены)

Структура на Google Drive:
```
MyDrive/
└── agricultural_segmentation/
    ├── data/
    │   ├── multiband_timeseries/
    │   ├── osm_masks/
    │   └── scl_masks/
    └── train_segmentation_advanced.py
```

In [7]:
import shutil
from pathlib import Path

# Путь к данным на Google Drive
GDRIVE_DATA = Path('/content/drive/MyDrive/agricultural_segmentation')

if GDRIVE_DATA.exists():
    print("✅ Данные найдены на Google Drive")

    # Скопировать данные в локальную директорию Colab (быстрее)
    print("Копирование данных...")
    !cp -r /content/drive/MyDrive/Agro_Seg_Classification* /content/agricultural_segmentation/
    print("✅ Данные скопированы")
else:
    print("❌ Данные не найдены на Google Drive")
    print("Загрузите данные по инструкции ниже")

✅ Данные найдены на Google Drive
Копирование данных...
✅ Данные скопированы


### Вариант B: Загрузить архив через Colab Files

1. Запакуйте локально:
   ```bash
   # На вашем компьютере
   python prepare_for_colab.py
   ```
   
2. Загрузите `agricultural_data.zip` через кнопку Files → Upload

In [9]:
# Разархивировать загруженный файл
import zipfile

zip_path = '/content/drive/MyDrive/Agro_Seg_Classification/agricultural_data.zip'

if Path(zip_path).exists():
    print("Разархивирование данных...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/agricultural_segmentation')
    print("✅ Данные разархивированы")
else:
    print("❌ Архив не найден. Загрузите agricultural_data.zip")

Разархивирование данных...
✅ Данные разархивированы


## 5. Проверка данных

In [10]:
import os

# Проверить структуру данных
data_dir = Path('data/multiband_timeseries')
osm_mask_dir = Path('data/osm_masks')
scl_mask_dir = Path('data/scl_masks')

print("Структура данных:")
print(f"  Multiband data: {'✅' if data_dir.exists() else '❌'} ({len(list(data_dir.rglob('*.npz'))) if data_dir.exists() else 0} файлов)")
print(f"  OSM masks: {'✅' if osm_mask_dir.exists() else '❌'} ({len(list(osm_mask_dir.rglob('*.npy'))) if osm_mask_dir.exists() else 0} файлов)")
print(f"  SCL masks: {'✅' if scl_mask_dir.exists() else '❌'} ({len(list(scl_mask_dir.rglob('*.npy'))) if scl_mask_dir.exists() else 0} файлов)")

if data_dir.exists():
    regions = [d.name for d in data_dir.iterdir() if d.is_dir()]
    print(f"\nРегионы: {regions}")

Структура данных:
  Multiband data: ✅ (1122 файлов)
  OSM masks: ✅ (811 файлов)
  SCL masks: ✅ (187 файлов)

Регионы: ['Krasnodar_Krai', 'Kursk', 'Stavropol', 'Rostov']


## 6. Загрузка кода модели

In [16]:
# Если train_segmentation_advanced.py НЕ загружен, создадим его здесь
# Или загрузите файл через Files → Upload

if not Path('/content/drive/MyDrive/Agro_Seg_Classification/train_segmentation_advanced.py').exists():
    print("⚠️ train_segmentation_advanced.py не найден")
    print("Загрузите файл через Files → Upload")
else:
    print("✅ train_segmentation_advanced.py найден")

✅ train_segmentation_advanced.py найден


## 7. Конфигурация для Colab (оптимизировано для GPU)

In [17]:
# Конфигурация оптимизирована для T4 GPU (16GB VRAM)
COLAB_CONFIG = {
    # Пути
    'data_dir': 'data/multiband_timeseries',
    'scl_mask_dir': 'data/scl_masks',
    'osm_mask_dir': 'data/osm_masks',
    'output_dir': 'models/segmentation',

    # Параметры данных
    'image_size': (256, 256),  # Можно увеличить до (512, 512) на A100
    'num_classes': 5,
    'use_cache': False,  # True если хватает RAM

    # Параметры обучения (оптимизировано для GPU)
    'batch_size': 128,  # T4: 16-32, A100: 64-128
    'num_epochs': 100,  # Меньше для быстрого теста
    'learning_rate': 0.001,
    'weight_decay': 1e-5,

    # Модель
    'use_attention': True,
    'in_channels': 10,

    # Loss
    'use_weighted_loss': True,
    'dice_weight': 0.5,

    # Оптимизация
    'use_amp': True,  # Обязательно для GPU!
    'early_stopping_patience': 10,

    # Валидация
    'val_split': 0.2,
    'random_seed': 42
}

print("Конфигурация для Colab:")
for key, value in COLAB_CONFIG.items():
    print(f"  {key}: {value}")

Конфигурация для Colab:
  data_dir: data/multiband_timeseries
  scl_mask_dir: data/scl_masks
  osm_mask_dir: data/osm_masks
  output_dir: models/segmentation
  image_size: (256, 256)
  num_classes: 5
  use_cache: False
  batch_size: 128
  num_epochs: 100
  learning_rate: 0.001
  weight_decay: 1e-05
  use_attention: True
  in_channels: 10
  use_weighted_loss: True
  dice_weight: 0.5
  use_amp: True
  early_stopping_patience: 10
  val_split: 0.2
  random_seed: 42


## 8. Запуск обучения

**⚠️ ВАЖНО:** Обучение может занять 1-3 часа на T4 GPU

In [ ]:
"""
Улучшенный скрипт для обучения модели сегментации сельскохозяйственных полей.

Новые возможности:
- Data augmentation (повороты, отражения, яркость, контраст)
- Расширенные метрики (Dice, Precision, Recall, F1)
- TensorBoard для мониторинга обучения
- Weighted loss для несбалансированных классов
- Сохранение чекпоинтов каждую эпоху
- Early stopping
- Поддержка mixed precision training (AMP)
- Валидация на отдельных регионах
- Поддержка реальных меток из OpenStreetMap
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from torch.amp import autocast, GradScaler
from pathlib import Path
import logging
from typing import Tuple, List, Dict, Optional
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
from datetime import datetime
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import os # Added os import

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler('training_segmentation.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# =====================================================================
# 1. Data Augmentation
# =====================================================================
def get_training_augmentation(image_size: Tuple[int, int] = (256, 256)):
    """
    Создает pipeline аугментации для обучения.

    Args:
        image_size: Размер выходного изображения
    """
    return A.Compose([
        A.RandomCrop(height=image_size[0], width=image_size[1], p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(
            translate_percent=0.1,
            scale=(0.8, 1.2),
            rotate=(-45, 45),
            p=0.5
        ),
        # Аугментация для спектральных каналов
        A.RandomBrightnessContrast(
            brightness_limit=0.2,
            contrast_limit=0.2,
            p=0.5
        ),
        # Corrected argument: use var_limit instead of variance_limit
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    ])


def get_validation_augmentation(image_size: Tuple[int, int] = (256, 256)):
    """Pipeline для валидации (только resize/crop)."""
    return A.Compose([
        A.CenterCrop(height=image_size[0], width=image_size[1], p=1.0), # Removed always_apply
    ])


# =====================================================================
# 2. Улучшенный Dataset
# =====================================================================
class AdvancedSegmentationDataset(Dataset):
    """Улучшенный dataset для сегментации полей с поддержкой аугментации."""

    def __init__(
        self,
        data_dir: Path,
        mask_dir: Optional[Path] = None,
        osm_mask_dir: Optional[Path] = None,
        transform=None,
        image_size: Tuple[int, int] = (256, 256),
        use_cache: bool = False
    ):
        """
        Args:
            data_dir: Путь к мультиспектральным снимкам
            mask_dir: Путь к маскам сегментации из SCL
            osm_mask_dir: Путь к маскам из OpenStreetMap (приоритет)
            transform: Albumentations transform
            image_size: Размер изображений
            use_cache: Кэширование данных в RAM
        """
        self.data_dir = data_dir
        self.mask_dir = mask_dir
        self.osm_mask_dir = osm_mask_dir
        self.transform = transform
        self.image_size = image_size
        self.use_cache = use_cache
        self.cache = {} if use_cache else None
        self.samples = []

        logger.info(f"Загрузка данных из {data_dir}")

        # Собрать все снимки
        for region_dir in data_dir.glob("*"):
            if not region_dir.is_dir():
                continue

            for tile_dir in region_dir.glob("tile_*"):
                for npz_file in tile_dir.glob("*.npz"):
                    self.samples.append({
                        'data_file': npz_file,
                        'region': region_dir.name,
                        'tile': tile_dir.name
                    })

        logger.info(f"Найдено {len(self.samples)} снимков")

        # Проверка масок
        self.use_osm_masks = osm_mask_dir is not None and osm_mask_dir.exists()
        self.use_scl_masks = mask_dir is not None and mask_dir.exists()

        if self.use_osm_masks:
            logger.info("Используются маски из OpenStreetMap")
        elif self.use_scl_masks:
            logger.info("Используются маски из SCL")
        else:
            logger.warning("Маски не найдены. Используются синтетические маски на основе NDVI.")

    def __len__(self):
        return len(self.samples)

    def _load_data(self, idx):
        """Загрузить данные (с кэшированием)."""
        if self.use_cache and idx in self.cache:
            return self.cache[idx]

        sample = self.samples[idx]
        npz_file = sample['data_file']
        region = sample['region']
        tile = sample['tile']

        data = np.load(npz_file, allow_pickle=True)

        # Загрузить мультиспектральные каналы
        if 'bands' not in data:
            raise ValueError(f"No 'bands' in {npz_file}")

        image = data['bands'].astype(np.float32)  # (H, W, C)

        # Обработать NaN
        image = np.nan_to_num(image, nan=0.0)

        # Нормализация (Sentinel-2 обычно до 10000)
        image = np.clip(image / 3000.0, 0, 1)

        # Загрузить маску
        mask = self._load_mask(npz_file, region, tile, image)

        result = (image, mask)

        if self.use_cache:
            self.cache[idx] = result

        return result

    def _load_mask(self, npz_file, region, tile, image):
        """Загрузить маску (OSM > SCL > synthetic)."""

        # 1. Попытка загрузить OSM маску
        if self.use_osm_masks:
            osm_mask_file = self.osm_mask_dir / region / tile / f"{npz_file.stem}_osm_mask.npy"
            if osm_mask_file.exists():
                mask = np.load(osm_mask_file)
                return mask

        # 2. Попытка загрузить SCL маску
        if self.use_scl_masks:
            scl_mask_file = self.mask_dir / region / tile / f"{npz_file.stem}_mask.npy"
            if scl_mask_file.exists():
                mask = np.load(scl_mask_file)
                return mask

        # 3. Создать синтетическую маску на основе NDVI
        return self._create_synthetic_mask(image)

    def _create_synthetic_mask(self, image):
        """Создать синтетическую маску на основе NDVI."""
        # BAND_ORDER = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']
        red = image[:, :, 2]  # B04
        nir = image[:, :, 6]  # B08

        ndvi = (nir - red) / (nir + red + 1e-6)
        ndvi = np.clip(ndvi, -1, 1)

        # Создать классы на основе NDVI
        mask = np.zeros(ndvi.shape, dtype=np.int64)
        mask[ndvi < 0.2] = 0  # Нет растительности / почва
        mask[(ndvi >= 0.2) & (ndvi < 0.4)] = 1  # Редкая растительность
        mask[(ndvi >= 0.4) & (ndvi < 0.6)] = 2  # Умеренная растительность
        mask[ndvi >= 0.6] = 3  # Густая растительность

        return mask

    def __getitem__(self, idx):
        image, mask = self._load_data(idx)

        # Apply augmentation
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        # Convert to torch tensors
        image = torch.FloatTensor(image).permute(2, 0, 1)  # (C, H, W)
        mask = torch.LongTensor(mask)  # (H, W)

        return image, mask


# =====================================================================
# 3. Улучшенная U-Net с Attention
# =====================================================================
class AttentionBlock(nn.Module):
    """Attention gate для U-Net."""

    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )

        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )

        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi


class DoubleConv(nn.Module):
    """(Conv => BN => ReLU) * 2"""

    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class AttentionUNet(nn.Module):
    """U-Net с Attention gates для лучшей сегментации."""

    def __init__(self, in_channels=10, num_classes=4, use_attention=True):
        super().__init__()
        self.use_attention = use_attention

        # Encoder
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(2)
        self.dropout = nn.Dropout2d(0.3)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Attention gates
        if use_attention:
            self.att4 = AttentionBlock(F_g=512, F_l=512, F_int=256)
            self.att3 = AttentionBlock(F_g=256, F_l=256, F_int=128)
            self.att2 = AttentionBlock(F_g=128, F_l=128, F_int=64)
            self.att1 = AttentionBlock(F_g=64, F_l=64, F_int=32)

        # Decoder
        self.upconv4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024, 512)

        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)

        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)

        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        # Output
        self.out = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool(enc1))
        enc3 = self.enc3(self.pool(enc2))
        enc4 = self.enc4(self.pool(enc3))

        # Bottleneck
        bottleneck = self.bottleneck(self.pool(enc4))
        bottleneck = self.dropout(bottleneck)

        # Decoder with skip connections and attention
        dec4 = self.upconv4(bottleneck)
        if self.use_attention:
            enc4 = self.att4(dec4, enc4)
        dec4 = torch.cat([dec4, enc4], dim=1)
        dec4 = self.dec4(dec4)

        dec3 = self.upconv3(dec4)
        if self.use_attention:
            enc3 = self.att3(dec3, enc3)
        dec3 = torch.cat([dec3, enc3], dim=1)
        dec3 = self.dec3(dec3)

        dec2 = self.upconv2(dec3)
        if self.use_attention:
            enc2 = self.att2(dec2, enc2)
        dec2 = torch.cat([dec2, enc2], dim=1)
        dec2 = self.dec2(dec2)

        dec1 = self.upconv1(dec2)
        if self.use_attention:
            enc1 = self.att1(dec1, enc1)
        dec1 = torch.cat([dec1, enc1], dim=1)
        dec1 = self.dec1(dec1)

        # Output
        out = self.out(dec1)

        return out


# =====================================================================
# 4. Улучшенные Loss функции
# =====================================================================
class DiceLoss(nn.Module):
    """Dice Loss для сегментации."""

    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred = F.softmax(pred, dim=1)

        # One-hot encode target
        target_one_hot = F.one_hot(target, num_classes=pred.shape[1])
        target_one_hot = target_one_hot.permute(0, 3, 1, 2).float()

        # Compute Dice
        intersection = (pred * target_one_hot).sum(dim=(2, 3))
        union = pred.sum(dim=(2, 3)) + target_one_hot.sum(dim=(2, 3))

        dice = (2. * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()


class CombinedLoss(nn.Module):
    """Комбинация Cross Entropy и Dice Loss."""

    def __init__(self, weight=None, dice_weight=0.5):
        super().__init__()
        self.ce_loss = nn.CrossEntropyLoss(weight=weight)
        self.dice_loss = DiceLoss()
        self.dice_weight = dice_weight

    def forward(self, pred, target):
        ce = self.ce_loss(pred, target)
        dice = self.dice_loss(pred, target)
        return (1 - self.dice_weight) * ce + self.dice_weight * dice


# =====================================================================
# 5. Расширенные метрики
# =====================================================================
class SegmentationMetrics:
    """Класс для вычисления метрик сегментации."""

    def __init__(self, num_classes):
        self.num_classes = num_classes
        self.reset()

    def reset(self):
        """Сброс накопленных значений."""
        self.confusion_matrix = np.zeros((self.num_classes, self.num_classes))

    def update(self, pred, target):
        """Обновить матрицу ошибок."""
        pred = pred.cpu().numpy().flatten()
        target = target.cpu().numpy().flatten()

        mask = (target >= 0) & (target < self.num_classes)
        pred = pred[mask]
        target = target[mask]

        cm = np.bincount(
            self.num_classes * target + pred,
            minlength=self.num_classes ** 2
        ).reshape(self.num_classes, self.num_classes)

        self.confusion_matrix += cm

    def get_metrics(self):
        """Вычислить все метрики."""
        cm = self.confusion_matrix

        # True Positives, False Positives, False Negatives
        tp = np.diag(cm)
        fp = cm.sum(axis=0) - tp
        fn = cm.sum(axis=1) - tp

        # Metrics per class
        iou = tp / (tp + fp + fn + 1e-10)
        precision = tp / (tp + fp + 1e-10)
        recall = tp / (tp + fn + 1e-10)
        f1 = 2 * (precision * recall) / (precision + recall + 1e-10)

        # Overall accuracy
        accuracy = tp.sum() / (cm.sum() + 1e-10)

        return {
            'iou_per_class': iou,
            'mean_iou': np.nanmean(iou),
            'precision_per_class': precision,
            'mean_precision': np.nanmean(precision),
            'recall_per_class': recall,
            'mean_f1': np.nanmean(f1),
            'accuracy': accuracy
        }


# =====================================================================
# 6. Trainer с TensorBoard
# =====================================================================
class SegmentationTrainer:
    """Класс для обучения модели сегментации."""

    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        scheduler, # Added scheduler parameter
        device,
        num_classes,
        output_dir='models/segmentation',
        use_amp=True,
        log_dir='runs/segmentation'
    ):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler # Assigned scheduler
        self.device = device
        self.num_classes = num_classes
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.use_amp = use_amp

        # TensorBoard
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.writer = SummaryWriter(f'{log_dir}/{timestamp}')

        # AMP scaler
        self.scaler = GradScaler(device.type) if use_amp and device.type == 'cuda' else None
        self.use_amp = use_amp and device.type == 'cuda'  # AMP только для CUDA

        # Metrics
        self.train_metrics = SegmentationMetrics(num_classes)
        self.val_metrics = SegmentationMetrics(num_classes)

        # History
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'val_mean_iou': [],
            'val_mean_f1': [],
            'learning_rate': []
        }

        # Best model tracking
        self.best_val_iou = 0.0
        self.patience_counter = 0

    def train_epoch(self, epoch):
        """Обучить одну эпоху."""
        self.model.train()
        self.train_metrics.reset()
        total_loss = 0

        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch} [Train]")

        for batch_idx, (images, masks) in enumerate(pbar):
            images = images.to(self.device)
            masks = masks.to(self.device)

            self.optimizer.zero_grad()

            # Mixed precision training
            if self.use_amp:
                with autocast('cuda'):
                    outputs = self.model(images)
                    loss = self.criterion(outputs, masks)

                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                outputs = self.model(images)
                loss = self.criterion(outputs, masks)
                loss.backward()
                self.optimizer.step()

            total_loss += loss.item()

            # Update metrics
            _, predicted = torch.max(outputs, 1)
            self.train_metrics.update(predicted, masks)

            # Update progress bar
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

            # Log to TensorBoard
            global_step = epoch * len(self.train_loader) + batch_idx
            self.writer.add_scalar('Train/BatchLoss', loss.item(), global_step)

        avg_loss = total_loss / len(self.train_loader)
        metrics = self.train_metrics.get_metrics()

        return avg_loss, metrics

    def validate(self, epoch):
        """Валидация модели."""
        self.model.eval()
        self.val_metrics.reset()
        total_loss = 0

        with torch.no_grad():
            pbar = tqdm(self.val_loader, desc=f"Epoch {epoch} [Val]")

            for images, masks in pbar:
                images = images.to(self.device)
                masks = masks.to(self.device)

                outputs = self.model(images)
                loss = self.criterion(outputs, masks)

                total_loss += loss.item()

                # Update metrics
                _, predicted = torch.max(outputs, 1)
                self.val_metrics.update(predicted, masks)

                pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        avg_loss = total_loss / len(self.val_loader)
        metrics = self.val_metrics.get_metrics()

        return avg_loss, metrics

    def log_metrics(self, epoch, train_loss, train_metrics, val_loss, val_metrics):
        """Логировать метрики в TensorBoard."""

        # Losses
        self.writer.add_scalars('Loss', {
            'train': train_loss,
            'val': val_loss
        }, epoch)

        # IoU
        self.writer.add_scalar('Metrics/Val_mIoU', val_metrics['mean_iou'], epoch)
        self.writer.add_scalar('Metrics/Val_mF1', val_metrics['mean_f1'], epoch)
        self.writer.add_scalar('Metrics/Val_Accuracy', val_metrics['accuracy'], epoch)

        # Per-class IoU
        for i in range(self.num_classes):
            self.writer.add_scalar(f'IoU/Class_{i}', val_metrics['iou_per_class'][i], epoch)

        # Learning Rate
        current_lr = self.optimizer.param_groups[0]['lr']
        self.writer.add_scalar('Learning_Rate', current_lr, epoch)

    def save_checkpoint(self, epoch, val_metrics, is_best=False):
        """Сохранить чекпоинт (только последний и лучший)."""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict() if self.scheduler else None, # Save scheduler state
            'val_metrics': val_metrics,
            'history': self.history
        }

        # Удалить предыдущий последний чекпоинт, если существует
        last_checkpoint_path = self.output_dir / 'last_checkpoint.pth'
        if last_checkpoint_path.exists():
            os.remove(last_checkpoint_path)

        # Сохранить последний чекпоинт
        torch.save(checkpoint, last_checkpoint_path)
        logger.info(f"✓ Last checkpoint saved to {last_checkpoint_path}")

        # Сохранить лучшую модель
        if is_best:
            best_path = self.output_dir / 'best_model.pth'
            torch.save(checkpoint, best_path)
            logger.info(f"✓ New best model saved! Val mIoU: {val_metrics['mean_iou']:.4f}")


    def train(self, num_epochs, early_stopping_patience=10):
        """Основной цикл обучения."""
        logger.info("="*70)
        logger.info(f"Starting training for {num_epochs} epochs")
        logger.info(f"Device: {self.device}")
        logger.info(f"AMP enabled: {self.use_amp}")
        logger.info("="*70)

        for epoch in range(1, num_epochs + 1):
            # Train
            train_loss, train_metrics = self.train_epoch(epoch)

            # Validate
            val_loss, val_metrics = self.validate(epoch)

            # Scheduler step (if scheduler exists)
            if self.scheduler:
                self.scheduler.step(val_metrics['mean_iou'])

            # Log metrics
            self.log_metrics(epoch, train_loss, train_metrics, val_loss, val_metrics)

            # Update history
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_mean_iou'].append(val_metrics['mean_iou'])
            self.history['val_mean_f1'].append(val_metrics['mean_f1'])
            self.history['learning_rate'].append(self.optimizer.param_groups[0]['lr'])

            # Print epoch summary
            logger.info(f"\nEpoch {epoch}/{num_epochs}")
            logger.info(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            logger.info(f"Val mIoU: {val_metrics['mean_iou']:.4f} | Val mF1: {val_metrics['mean_f1']:.4f}")
            logger.info(f"Val Accuracy: {val_metrics['accuracy']:.4f}")
            logger.info(f"Per-class IoU: {val_metrics['iou_per_class']}")

            # Check for best model
            is_best = val_metrics['mean_iou'] > self.best_val_iou
            if is_best:
                self.best_val_iou = val_metrics['mean_iou']
                self.patience_counter = 0
            else:
                self.patience_counter += 1

            # Save checkpoint
            self.save_checkpoint(epoch, val_metrics, is_best)

            # Early stopping
            if self.patience_counter >= early_stopping_patience:
                logger.info(f"\nEarly stopping triggered after {epoch} epochs")
                break

        self.writer.close()
        logger.info("="*70)
        logger.info(f"Training complete! Best Val mIoU: {self.best_val_iou:.4f}")
        logger.info("="*70)

        return self.history


# =====================================================================
# 7. Визуализация
# =====================================================================
def visualize_predictions(model, dataset, device, num_samples=8, save_path='visualizations/predictions.png'):
    """Визуализировать предсказания модели."""
    model.eval()

    num_rows = (num_samples + 3) // 4
    fig, axes = plt.subplots(num_rows, 12, figsize=(30, 7 * num_rows))
    if num_rows == 1:
        axes = axes.reshape(1, -1)

    class_names = ['Invalid', 'Bare Soil', 'Sparse Veg', 'Moderate Veg', 'Dense Veg']

    for i in range(num_samples):
        row = i // 4
        col_base = (i % 4) * 3

        idx = np.random.randint(len(dataset))

        # Get image and mask tensors directly from the dataset's __getitem__
        image_tensor, mask_tensor = dataset[idx]

        with torch.no_grad():
            image_batch = image_tensor.unsqueeze(0).to(device)
            output = model(image_batch)
            _, predicted = torch.max(output, 1)
            predicted = predicted.squeeze(0).cpu().numpy()


        # RGB composite (B04, B03, B02)
        rgb_indices = [2, 1, 0]
        # Use the transformed image tensor for RGB
        rgb = image_tensor[rgb_indices, :, :].permute(1, 2, 0).numpy()
        rgb = np.clip(rgb, 0, 1)

        # Ground truth
        gt_mask = mask_tensor.numpy()

        # Plot RGB
        axes[row, col_base].imshow(rgb)
        axes[row, col_base].set_title('RGB', fontsize=10)
        axes[row, col_base].axis('off')

        # Plot GT
        axes[row, col_base + 1].imshow(gt_mask, cmap='tab10', vmin=0, vmax=4)
        axes[row, col_base + 1].set_title('Ground Truth', fontsize=10)
        axes[row, col_base + 1].axis('off')

        # Plot Prediction
        axes[row, col_base + 2].imshow(predicted, cmap='tab10', vmin=0, vmax=4)
        axes[row, col_base + 2].set_title('Prediction', fontsize=10)
        axes[row, col_base + 2].axis('off')

    plt.tight_layout()
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

    logger.info(f"Predictions saved to {save_path}")


# =====================================================================
# 8. Главная функция
# =====================================================================
def main():
    # ======================== КОНФИГУРАЦИЯ ========================
    # Load config from file if it exists, otherwise use default
    config_path = Path('config_colab.json')
    if config_path.exists():
        logger.info(f"Loading configuration from {config_path}")
        with open(config_path, 'r') as f:
            config = json.load(f)
    else:
        logger.warning("Configuration file not found. Using default configuration.")
        config = {
            # Пути
            'data_dir': 'data/multiband_timeseries',
            'scl_mask_dir': 'data/scl_masks',
            'osm_mask_dir': 'data/osm_masks',  # Опционально
            'output_dir': 'models/segmentation',

            # Параметры данных
            'image_size': (256, 256),
            'num_classes': 5,  # 0=invalid, 1=bare soil, 2=sparse, 3=moderate, 4=dense
            'use_cache': False,

            # Параметры обучения
            'batch_size': 16,
            'num_epochs': 100,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,

            # Модель
            'use_attention': True,
            'in_channels': 10,

            # Loss
            'use_weighted_loss': True,
            'dice_weight': 0.5,

            # Оптимизация
            'use_amp': True,
            'early_stopping_patience': 15,

            # Валидация
            'val_split': 0.2,
            'random_seed': 42
        }


    # Установка random seed
    torch.manual_seed(config['random_seed'])
    np.random.seed(config['random_seed'])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Device: {device}")

    # ======================== ЗАГРУЗКА ДАННЫХ ========================
    logger.info("Loading datasets...")

    train_transform = get_training_augmentation(config['image_size'])
    val_transform = get_validation_augmentation(config['image_size'])

    # Check if mask directories exist and pass None if they don't
    scl_mask_dir = Path(config['scl_mask_dir']) if Path(config['scl_mask_dir']).exists() else None
    osm_mask_dir = Path(config['osm_mask_dir']) if Path(config['osm_mask_dir']).exists() else None

    full_dataset = AdvancedSegmentationDataset(
        data_dir=Path(config['data_dir']),
        mask_dir=scl_mask_dir,
        osm_mask_dir=osm_mask_dir,
        transform=None, # Apply transforms later
        image_size=config['image_size'],
        use_cache=config['use_cache']
    )

    # Split train/val
    train_size = int((1 - config['val_split']) * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(config['random_seed'])
    )

    # Assign transforms to the underlying dataset of the subsets
    train_dataset.dataset.transform = train_transform
    val_dataset.dataset.transform = val_transform


    train_loader = DataLoader(
        train_dataset,
        batch_size=config['batch_size'],
        shuffle=True,
        num_workers=4,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    logger.info(f"Train samples: {train_size}, Val samples: {val_size}")

    # ======================== СОЗДАНИЕ МОДЕЛИ ========================
    logger.info("Creating model...")

    model = AttentionUNet(
        in_channels=config['in_channels'],
        num_classes=config['num_classes'],
        use_attention=config['use_attention']
    ).to(device)

    logger.info(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    # ======================== LOSS & OPTIMIZER ========================

    # Вычислить веса классов (если используется weighted loss)
    if config['use_weighted_loss']:
        logger.info("Computing class weights...")
        class_counts = np.zeros(config['num_classes'])

        for _, mask in tqdm(train_dataset, desc="Computing weights"):
            mask_np = mask.numpy()
            for c in range(config['num_classes']):
                class_counts[c] += (mask_np == c).sum()

        # Inverse frequency weighting
        class_weights = 1.0 / (class_counts + 1)
        class_weights = class_weights / class_weights.sum() * config['num_classes']
        class_weights = torch.FloatTensor(class_weights).to(device)

        logger.info(f"Class weights: {class_weights}")
    else:
        class_weights = None

    criterion = CombinedLoss(
        weight=class_weights,
        dice_weight=config['dice_weight']
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )

    # Add Learning Rate Scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max', # Monitor validation metric (e.g., mIoU)
        factor=0.5, # Reduce LR by half
        patience=5, # Number of epochs with no improvement after which learning rate will be reduced.
        # verbose=True # Removed verbose=True
    )


    # ======================== ОБУЧЕНИЕ ========================
    trainer = SegmentationTrainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler, # Pass the scheduler to the trainer
        device=device,
        num_classes=config['num_classes'],
        output_dir=config['output_dir'],
        use_amp=config['use_amp']
    )

    history = trainer.train(
        num_epochs=config['num_epochs'],
        early_stopping_patience=config['early_stopping_patience']
    )

    # ======================== ВИЗУАЛИЗАЦИЯ ========================
    logger.info("Creating visualizations...")

    # Load best model
    best_model_path = Path(config['output_dir']) / 'best_model.pth'
    if best_model_path.exists(): # Check if best_model.pth exists
        # Modified to load with weights_only=False
        checkpoint = torch.load(best_model_path, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])

        # Visualize predictions
        visualize_predictions(
            model,
            # Pass the validation subset here
            val_dataset,
            device,
            num_samples=8,
            save_path=f"{config['output_dir']}/predictions.png"
        )

        # Plot training history
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))

        # Loss
        axes[0, 0].plot(history['train_loss'], label='Train')
        axes[0, 0].plot(history['val_loss'], label='Val')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].set_title('Training & Validation Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True)

        # mIoU
        axes[0, 1].plot(history['val_mean_iou'])
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Mean IoU')
        axes[0, 1].set_title('Validation Mean IoU')
        axes[0, 1].grid(True)

        # mF1
        axes[1, 0].plot(history['val_mean_f1'])
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Mean F1')
        axes[1, 0].set_title('Validation Mean F1')
        axes[1, 0].grid(True)

        # Learning Rate
        axes[1, 1].plot(history['learning_rate'])
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Learning Rate')
        axes[1, 1].set_title('Learning Rate Schedule')
        axes[1, 1].set_yscale('log')
        axes[1, 1].grid(True)

        plt.tight_layout()
        plt.savefig(f"{config['output_dir']}/training_history.png", dpi=150)
        plt.close()

        # Save config
        with open(f"{config['output_dir']}/config.json", 'w') as f:
            json.dump(config, f, indent=2)

        logger.info(f"\n✓ Training complete! Models saved to {config['output_dir']}")
        logger.info(f"✓ Best validation mIoU: {trainer.best_val_iou:.4f}")
    else:
        logger.warning("Best model checkpoint not found. Skipping visualization.")


if __name__ == "__main__":
    main()

Epoch 5 [Train]:   0%|          | 0/8 [00:04<?, ?it/s]


KeyboardInterrupt: 

In [18]:
# Импорт необходимых модулей
import sys
import json

# Сохранить конфигурацию
with open('config_colab.json', 'w') as f:
    json.dump(COLAB_CONFIG, f, indent=2)

# Запустить обучение
# Вариант 1: Если train_segmentation_advanced.py загружен
if Path('train_segmentation_advanced.py').exists():
    # Модифицируем конфигурацию в скрипте
    !python train_segmentation_advanced.py
else:
    # Вариант 2: Запустить код напрямую (если скрипт встроен в notebook)
    print("Загрузите train_segmentation_advanced.py")

2025-10-12 18:37:12.915546: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-12 18:37:12.932944: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760294232.954015    3976 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760294232.960435    3976 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760294232.976763    3976 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## 9. Мониторинг обучения через TensorBoard

In [ ]:
# Загрузить TensorBoard в Colab
%load_ext tensorboard
%tensorboard --logdir runs/segmentation

## 10. Сохранение результатов на Google Drive

In [ ]:
import shutil
from datetime import datetime

# Создать папку для результатов
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_dir = f'/content/drive/MyDrive/agricultural_segmentation/results_{timestamp}'
os.makedirs(results_dir, exist_ok=True)

# Скопировать модели
if Path('models/segmentation').exists():
    print("Сохранение моделей...")
    shutil.copytree('models/segmentation', f'{results_dir}/models', dirs_exist_ok=True)
    print("✅ Модели сохранены")

# Скопировать TensorBoard логи
if Path('runs/segmentation').exists():
    print("Сохранение TensorBoard логов...")
    shutil.copytree('runs/segmentation', f'{results_dir}/tensorboard', dirs_exist_ok=True)
    print("✅ Логи сохранены")

# Скопировать визуализации
if Path('visualizations').exists():
    print("Сохранение визуализаций...")
    shutil.copytree('visualizations', f'{results_dir}/visualizations', dirs_exist_ok=True)
    print("✅ Визуализации сохранены")

print(f"\n✅ Все результаты сохранены в:\n{results_dir}")

## 11. Визуализация результатов

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt

# Показать предсказания
predictions_path = 'models/segmentation/predictions.png'
if Path(predictions_path).exists():
    print("Предсказания модели:")
    display(Image(predictions_path))

# Показать графики обучения
history_path = 'models/segmentation/training_history.png'
if Path(history_path).exists():
    print("\nИстория обучения:")
    display(Image(history_path))

## 12. Загрузка обученной модели для inference

In [ ]:
import torch
import numpy as np
from pathlib import Path

# Загрузить лучшую модель
checkpoint = torch.load('models/segmentation/best_model.pth')

# Вывести метрики
val_metrics = checkpoint['val_metrics']
print("Метрики лучшей модели:")
print(f"  mIoU: {val_metrics['mean_iou']:.4f}")
print(f"  mF1: {val_metrics['mean_f1']:.4f}")
print(f"  Accuracy: {val_metrics['accuracy']:.4f}")
print(f"  Per-class IoU: {val_metrics['iou_per_class']}")

## 🎉 Отлично! Модель обучена!

Теперь переходим к детальной визуализации результатов.

В следующих секциях мы:
1. Загрузим обученную модель
2. Создадим детальные визуализации предсказаний
3. Сравним с ground truth масками
4. Найдем лучшие и худшие примеры
5. Построим confusion matrix и per-class метрики

### Полезные ссылки:
- [TensorBoard](https://tensorboard.dev/) - публичный хостинг логов
- [Colab Pro](https://colab.research.google.com/signup) - больше GPU времени и лучшие GPU

## 🎊 Готово! Визуализации созданы!

### Что вы получили:

1. **Детальные визуализации** - каждое изображение показывает:
   - RGB композит (True Color)
   - NDVI карта
   - Ground Truth маска
   - Предсказание модели
   - Оверлеи на RGB изображении
   - Карта ошибок (красным)
   - Карта уверенности модели
   - IoU по каждому классу
   - Confusion Matrix

2. **Метрики для каждого примера**:
   - Accuracy (точность)
   - mIoU (mean Intersection over Union)
   - Per-class IoU

3. **Все результаты сохранены**:
   - На Google Drive в папке `results_YYYYMMDD_HHMMSS/visualizations/`
   - Можете скачать и использовать для отчетов

### Интерпретация результатов:

- **mIoU > 0.7** ✅ - отличное качество сегментации
- **mIoU 0.5-0.7** 🟡 - хорошее качество
- **mIoU < 0.5** ❌ - требуется улучшение

**Красные области на Difference Map** - ошибки модели (где предсказание не совпадает с GT)

**Confidence Map** - насколько модель уверена в своих предсказаниях (1.0 = 100% уверенность)

In [ ]:
# Создать папку для визуализаций
viz_dir = Path('visualizations/colab_results')
viz_dir.mkdir(parents=True, exist_ok=True)

# 1. Сохранить легенду
print("Сохранение легенды классов...")
fig, ax = plt.subplots(figsize=(8, 6))
patches = []
for i, (name, color) in enumerate(zip(visualizer.class_names, visualizer.class_colors)):
    patches.append(mpatches.Patch(
        color=color / 255,
        label=f'Class {i}: {name}'
    ))
ax.legend(handles=patches, loc='center', fontsize=14, frameon=True)
ax.axis('off')
ax.set_title('Segmentation Classes', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(viz_dir / 'class_legend.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Легенда сохранена")

# 2. Сохранить детальные примеры
print("\nСохранение детальных примеров...")
num_detailed = 5
sample_indices = np.random.choice(len(val_dataset), num_detailed, replace=False)

for i, idx in enumerate(sample_indices):
    print(f"  Пример {i+1}/{num_detailed}...")

    image, mask_gt, mask_pred, logits = visualizer.predict(idx)

    # RGB
    rgb_indices = [2, 1, 0]
    rgb = image[rgb_indices, :, :].permute(1, 2, 0).numpy()
    rgb = np.clip(rgb, 0, 1)

    # NDVI
    red = image[2, :, :].numpy()
    nir = image[6, :, :].numpy()
    ndvi = (nir - red) / (nir + red + 1e-6)
    ndvi = np.clip(ndvi, -1, 1)

    # Маски
    mask_gt_np = mask_gt.numpy()
    mask_pred_np = mask_pred.numpy()
    mask_gt_rgb = visualizer.mask_to_rgb(mask_gt_np)
    mask_pred_rgb = visualizer.mask_to_rgb(mask_pred_np)

    # Difference и confidence
    difference = (mask_gt_np != mask_pred_np).astype(int)
    confidence = F.softmax(logits, dim=0).max(dim=0)[0].numpy()

    # Метрики
    metrics = visualizer.compute_metrics(mask_gt, mask_pred)

    # Создать фигуру
    fig = plt.figure(figsize=(20, 12))
    gs = GridSpec(3, 4, figure=fig, hspace=0.3, wspace=0.3)

    # Заполнить все subplots (копия из visualize_single_prediction)
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(rgb)
    ax1.set_title('RGB Image', fontsize=12, fontweight='bold')
    ax1.axis('off')

    ax2 = fig.add_subplot(gs[0, 1])
    im2 = ax2.imshow(ndvi, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
    ax2.set_title('NDVI', fontsize=12, fontweight='bold')
    ax2.axis('off')
    plt.colorbar(im2, ax=ax2, fraction=0.046)

    ax3 = fig.add_subplot(gs[0, 2])
    ax3.imshow(mask_gt_rgb)
    ax3.set_title('Ground Truth', fontsize=12, fontweight='bold')
    ax3.axis('off')

    ax4 = fig.add_subplot(gs[0, 3])
    ax4.imshow(mask_pred_rgb)
    ax4.set_title('Prediction', fontsize=12, fontweight='bold')
    ax4.axis('off')

    ax5 = fig.add_subplot(gs[1, 0])
    ax5.imshow(rgb)
    ax5.imshow(mask_gt_rgb, alpha=0.5)
    ax5.set_title('GT Overlay', fontsize=12, fontweight='bold')
    ax5.axis('off')

    ax6 = fig.add_subplot(gs[1, 1])
    ax6.imshow(rgb)
    ax6.imshow(mask_pred_rgb, alpha=0.5)
    ax6.set_title('Prediction Overlay', fontsize=12, fontweight='bold')
    ax6.axis('off')

    ax7 = fig.add_subplot(gs[1, 2])
    im7 = ax7.imshow(difference, cmap='Reds', vmin=0, vmax=1)
    ax7.set_title(f'Errors\\nAcc: {metrics["accuracy"]:.2%}', fontsize=12, fontweight='bold')
    ax7.axis('off')
    plt.colorbar(im7, ax=ax7, fraction=0.046)

    ax8 = fig.add_subplot(gs[1, 3])
    im8 = ax8.imshow(confidence, cmap='viridis', vmin=0, vmax=1)
    ax8.set_title(f'Confidence\\nMean: {confidence.mean():.2%}', fontsize=12, fontweight='bold')
    ax8.axis('off')
    plt.colorbar(im8, ax=ax8, fraction=0.046)

    ax9 = fig.add_subplot(gs[2, 0:2])
    x = np.arange(len(visualizer.class_names))
    bars = ax9.bar(x, metrics['iou_per_class'], alpha=0.7)
    for j, bar in enumerate(bars):
        bar.set_color(visualizer.class_colors[j] / 255)
        bar.set_edgecolor('black')
        bar.set_linewidth(1)
    ax9.set_xticks(x)
    ax9.set_xticklabels(visualizer.class_names, rotation=45, ha='right')
    ax9.set_ylabel('IoU', fontsize=11)
    ax9.set_title(f'IoU per Class (mIoU: {metrics["mean_iou"]:.3f})', fontsize=12, fontweight='bold')
    ax9.set_ylim(0, 1)
    ax9.grid(True, alpha=0.3, axis='y')
    ax9.axhline(y=metrics['mean_iou'], color='red', linestyle='--', linewidth=2)

    ax10 = fig.add_subplot(gs[2, 2:])
    cm = visualizer._compute_confusion_matrix(mask_gt_np, mask_pred_np)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax10,
               xticklabels=[c[:10] for c in visualizer.class_names],
               yticklabels=[c[:10] for c in visualizer.class_names])
    ax10.set_title('Confusion Matrix', fontsize=12, fontweight='bold')
    ax10.set_xlabel('Predicted')
    ax10.set_ylabel('Ground Truth')

    fig.suptitle(f'Sample #{idx} | Accuracy: {metrics["accuracy"]:.2%} | mIoU: {metrics["mean_iou"]:.3f}',
                fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig(viz_dir / f'detailed_{i}_{idx}.png', dpi=150, bbox_inches='tight')
    plt.close()

print("✅ Детальные примеры сохранены")

# 3. Копировать на Google Drive
print("\nКопирование на Google Drive...")
drive_viz_dir = Path(results_dir) / 'visualizations'
if viz_dir.exists():
    shutil.copytree(viz_dir, drive_viz_dir, dirs_exist_ok=True)
    print(f"✅ Визуализации скопированы в:\\n{drive_viz_dir}")

    # Список файлов
    viz_files = list(drive_viz_dir.glob('*.png'))
    print(f"\\n📁 Сохранено {len(viz_files)} файлов:")
    for f in viz_files:
        print(f"  - {f.name}")

## 20. Сохранить визуализации на Google Drive

Сохраним все визуализации в папку результатов на Google Drive

In [ ]:
# Компактная визуализация 8 примеров
visualizer.visualize_multiple_samples(num_samples=8)

## 19. Компактная визуализация нескольких примеров

Покажем 8 примеров в компактном виде: RGB → GT → Prediction → Errors

In [ ]:
# Визуализировать несколько примеров
num_detailed = 3
sample_indices = np.random.choice(len(val_dataset), num_detailed, replace=False)

for i, idx in enumerate(sample_indices):
    print(f"\n{'='*60}")
    print(f"Пример {i+1}/{num_detailed} (sample #{idx})")
    print('='*60)
    metrics = visualizer.visualize_single_prediction(idx)

## 18. Еще несколько детальных примеров

Визуализируем еще 3-5 случайных примеров для более полного анализа

In [ ]:
# Визуализировать случайный пример
sample_idx = np.random.randint(0, len(val_dataset))
print(f"Визуализация примера #{sample_idx}")

metrics = visualizer.visualize_single_prediction(sample_idx)

## 17. Детальная визуализация одного примера

Покажем RGB, NDVI, Ground Truth, Prediction, Overlays, Difference Map, Confidence, IoU per class и Confusion Matrix

In [ ]:
# Показать легенду классов
visualizer.create_class_legend()

## 16. Легенда классов

In [ ]:
# Создать visualizer
visualizer = ColabModelVisualizer(
    model=model,
    dataset=val_dataset,
    device=device
)

print("✅ Visualizer готов к работе!")

In [ ]:
# Создать валидационный датасет
val_dataset = AdvancedSegmentationDataset(
    data_dir=Path('data/multiband_timeseries'),
    mask_dir=Path('data/scl_masks') if Path('data/scl_masks').exists() else None,
    osm_mask_dir=Path('data/osm_masks') if Path('data/osm_masks').exists() else None,
    transform=get_validation_augmentation((256, 256)),
    image_size=(256, 256),
    use_cache=False
)

print(f"✅ Загружено {len(val_dataset)} примеров для визуализации")

In [ ]:
# Загрузить обученную модель
checkpoint = torch.load('models/segmentation/best_model.pth', map_location=device)

# Восстановить модель
model = AttentionUNet(
    in_channels=10,
    num_classes=5,
    use_attention=True
).to(device)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"✅ Модель загружена")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Вывести метрики
if 'val_metrics' in checkpoint:
    metrics = checkpoint['val_metrics']
    print(f"\n📊 Метрики модели:")
    print(f"  mIoU: {metrics['mean_iou']:.4f}")
    print(f"  mF1: {metrics['mean_f1']:.4f}")
    print(f"  Accuracy: {metrics['accuracy']:.4f}")
    print(f"  Per-class IoU: {metrics['iou_per_class']}")

## 15. Загрузить валидационный датасет

In [ ]:
class ColabModelVisualizer:
    """Класс для визуализации результатов модели в Colab."""

    def __init__(self, model, dataset, device):
        self.model = model
        self.dataset = dataset
        self.device = device

        # Названия классов
        self.class_names = [
            'Invalid/Background',
            'Bare Soil',
            'Sparse Vegetation',
            'Moderate Vegetation',
            'Dense Vegetation'
        ]

        # Цвета для классов
        self.class_colors = np.array([
            [0, 0, 0],          # 0: Invalid - черный
            [139, 69, 19],      # 1: Bare soil - коричневый
            [255, 255, 0],      # 2: Sparse veg - желтый
            [144, 238, 144],    # 3: Moderate veg - светло-зеленый
            [0, 100, 0]         # 4: Dense veg - темно-зеленый
        ])

    def predict(self, idx):
        """Получить предсказание для одного изображения."""
        image, mask_gt = self.dataset[idx]

        with torch.no_grad():
            image_batch = image.unsqueeze(0).to(self.device)
            output = self.model(image_batch)
            _, predicted = torch.max(output, 1)
            predicted = predicted.squeeze(0).cpu()

        return image, mask_gt, predicted, output.squeeze(0).cpu()

    def compute_metrics(self, mask_gt, mask_pred, num_classes=5):
        """Вычислить метрики для одного примера."""
        metrics = {}

        mask_gt_np = mask_gt.numpy().flatten()
        mask_pred_np = mask_pred.numpy().flatten()

        # Accuracy
        metrics['accuracy'] = (mask_gt_np == mask_pred_np).mean()

        # Per-class metrics
        iou_per_class = []

        for c in range(num_classes):
            gt_c = (mask_gt_np == c)
            pred_c = (mask_pred_np == c)

            tp = (gt_c & pred_c).sum()
            fp = (~gt_c & pred_c).sum()
            fn = (gt_c & ~pred_c).sum()

            iou = tp / (tp + fp + fn + 1e-10)
            iou_per_class.append(iou)

        metrics['iou_per_class'] = np.array(iou_per_class)
        metrics['mean_iou'] = np.nanmean(iou_per_class)

        return metrics

    def mask_to_rgb(self, mask):
        """Конвертировать маску в RGB изображение."""
        rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
        for class_id, color in enumerate(self.class_colors):
            rgb[mask == class_id] = color
        return rgb

    def visualize_single_prediction(self, idx):
        """Детальная визуализация одного предсказания."""
        image, mask_gt, mask_pred, logits = self.predict(idx)

        # RGB композит
        rgb_indices = [2, 1, 0]  # B04, B03, B02
        rgb = image[rgb_indices, :, :].permute(1, 2, 0).numpy()
        rgb = np.clip(rgb, 0, 1)

        # NDVI
        red = image[2, :, :].numpy()
        nir = image[6, :, :].numpy()
        ndvi = (nir - red) / (nir + red + 1e-6)
        ndvi = np.clip(ndvi, -1, 1)

        # Маски
        mask_gt_np = mask_gt.numpy()
        mask_pred_np = mask_pred.numpy()

        # Difference map
        difference = (mask_gt_np != mask_pred_np).astype(int)

        # Confidence map
        confidence = F.softmax(logits, dim=0).max(dim=0)[0].numpy()

        # Вычислить метрики
        metrics = self.compute_metrics(mask_gt, mask_pred)

        # Создать фигуру
        fig = plt.figure(figsize=(20, 12))
        gs = GridSpec(3, 4, figure=fig, hspace=0.3, wspace=0.3)

        # 1. RGB изображение
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.imshow(rgb)
        ax1.set_title('RGB Image (True Color)', fontsize=12, fontweight='bold')
        ax1.axis('off')

        # 2. NDVI
        ax2 = fig.add_subplot(gs[0, 1])
        im2 = ax2.imshow(ndvi, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
        ax2.set_title('NDVI', fontsize=12, fontweight='bold')
        ax2.axis('off')
        plt.colorbar(im2, ax=ax2, fraction=0.046)

        # 3. Ground Truth
        ax3 = fig.add_subplot(gs[0, 2])
        mask_gt_rgb = self.mask_to_rgb(mask_gt_np)
        ax3.imshow(mask_gt_rgb)
        ax3.set_title('Ground Truth', fontsize=12, fontweight='bold')
        ax3.axis('off')

        # 4. Prediction
        ax4 = fig.add_subplot(gs[0, 3])
        mask_pred_rgb = self.mask_to_rgb(mask_pred_np)
        ax4.imshow(mask_pred_rgb)
        ax4.set_title('Prediction', fontsize=12, fontweight='bold')
        ax4.axis('off')

        # 5. Overlay GT на RGB
        ax5 = fig.add_subplot(gs[1, 0])
        ax5.imshow(rgb)
        ax5.imshow(mask_gt_rgb, alpha=0.5)
        ax5.set_title('GT Overlay', fontsize=12, fontweight='bold')
        ax5.axis('off')

        # 6. Overlay Prediction на RGB
        ax6 = fig.add_subplot(gs[1, 1])
        ax6.imshow(rgb)
        ax6.imshow(mask_pred_rgb, alpha=0.5)
        ax6.set_title('Prediction Overlay', fontsize=12, fontweight='bold')
        ax6.axis('off')

        # 7. Difference map
        ax7 = fig.add_subplot(gs[1, 2])
        im7 = ax7.imshow(difference, cmap='Reds', vmin=0, vmax=1)
        ax7.set_title(f'Errors (Red)\\nAccuracy: {metrics["accuracy"]:.2%}',
                     fontsize=12, fontweight='bold')
        ax7.axis('off')
        plt.colorbar(im7, ax=ax7, fraction=0.046)

        # 8. Confidence map
        ax8 = fig.add_subplot(gs[1, 3])
        im8 = ax8.imshow(confidence, cmap='viridis', vmin=0, vmax=1)
        ax8.set_title(f'Confidence\\nMean: {confidence.mean():.2%}',
                     fontsize=12, fontweight='bold')
        ax8.axis('off')
        plt.colorbar(im8, ax=ax8, fraction=0.046)

        # 9. Per-class IoU
        ax9 = fig.add_subplot(gs[2, 0:2])
        x = np.arange(len(self.class_names))
        bars = ax9.bar(x, metrics['iou_per_class'], alpha=0.7)

        # Раскрасить бары по классам
        for i, bar in enumerate(bars):
            bar.set_color(self.class_colors[i] / 255)
            bar.set_edgecolor('black')
            bar.set_linewidth(1)

        ax9.set_xticks(x)
        ax9.set_xticklabels(self.class_names, rotation=45, ha='right')
        ax9.set_ylabel('IoU', fontsize=11)
        ax9.set_title(f'IoU per Class (mIoU: {metrics["mean_iou"]:.3f})',
                     fontsize=12, fontweight='bold')
        ax9.set_ylim(0, 1)
        ax9.grid(True, alpha=0.3, axis='y')
        ax9.axhline(y=metrics['mean_iou'], color='red', linestyle='--',
                   linewidth=2, label=f'Mean IoU: {metrics["mean_iou"]:.3f}')
        ax9.legend()

        # 10. Confusion matrix
        ax10 = fig.add_subplot(gs[2, 2:])
        cm = self._compute_confusion_matrix(mask_gt_np, mask_pred_np)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax10,
                   xticklabels=[c[:10] for c in self.class_names],
                   yticklabels=[c[:10] for c in self.class_names],
                   cbar_kws={'label': 'Pixel Count'})
        ax10.set_title('Confusion Matrix', fontsize=12, fontweight='bold')
        ax10.set_xlabel('Predicted')
        ax10.set_ylabel('Ground Truth')

        # Общий заголовок
        fig.suptitle(f'Sample #{idx} | Accuracy: {metrics["accuracy"]:.2%} | mIoU: {metrics["mean_iou"]:.3f}',
                    fontsize=14, fontweight='bold')

        plt.tight_layout()
        plt.show()

        return metrics

    def _compute_confusion_matrix(self, mask_gt, mask_pred):
        """Вычислить confusion matrix."""
        num_classes = len(self.class_names)
        cm = np.zeros((num_classes, num_classes), dtype=int)

        for gt_class in range(num_classes):
            for pred_class in range(num_classes):
                cm[gt_class, pred_class] = ((mask_gt == gt_class) & (mask_pred == pred_class)).sum()

        return cm

    def visualize_multiple_samples(self, num_samples=8):
        """Компактная визуализация нескольких примеров."""
        # Выбрать случайные индексы
        sample_indices = np.random.choice(len(self.dataset), num_samples, replace=False)

        num_rows = num_samples
        fig, axes = plt.subplots(num_rows, 4, figsize=(16, 4 * num_rows))

        if num_rows == 1:
            axes = axes.reshape(1, -1)

        for i, idx in enumerate(sample_indices):
            image, mask_gt, mask_pred, _ = self.predict(idx)

            # RGB
            rgb_indices = [2, 1, 0]
            rgb = image[rgb_indices, :, :].permute(1, 2, 0).numpy()
            rgb = np.clip(rgb, 0, 1)

            # Маски
            mask_gt_np = mask_gt.numpy()
            mask_pred_np = mask_pred.numpy()
            mask_gt_rgb = self.mask_to_rgb(mask_gt_np)
            mask_pred_rgb = self.mask_to_rgb(mask_pred_np)

            # Difference
            difference = (mask_gt_np != mask_pred_np).astype(float)

            # Метрики
            metrics = self.compute_metrics(mask_gt, mask_pred)

            # Визуализация
            axes[i, 0].imshow(rgb)
            axes[i, 0].set_title(f'RGB #{idx}', fontsize=10)
            axes[i, 0].axis('off')

            axes[i, 1].imshow(mask_gt_rgb)
            axes[i, 1].set_title('Ground Truth', fontsize=10)
            axes[i, 1].axis('off')

            axes[i, 2].imshow(mask_pred_rgb)
            axes[i, 2].set_title(f'Prediction\\nmIoU: {metrics["mean_iou"]:.3f}',
                               fontsize=10)
            axes[i, 2].axis('off')

            axes[i, 3].imshow(difference, cmap='Reds', vmin=0, vmax=1)
            axes[i, 3].set_title(f'Errors\\nAcc: {metrics["accuracy"]:.2%}',
                               fontsize=10)
            axes[i, 3].axis('off')

        plt.tight_layout()
        plt.show()

    def create_class_legend(self):
        """Создать легенду классов."""
        fig, ax = plt.subplots(figsize=(8, 6))

        # Создать патчи для легенды
        patches = []
        for i, (name, color) in enumerate(zip(self.class_names, self.class_colors)):
            patches.append(mpatches.Patch(
                color=color / 255,
                label=f'Class {i}: {name}'
            ))

        ax.legend(handles=patches, loc='center', fontsize=14, frameon=True)
        ax.axis('off')
        ax.set_title('Segmentation Classes', fontsize=16, fontweight='bold', pad=20)

        plt.tight_layout()
        plt.show()

print("✅ ColabModelVisualizer создан")

## 14. Класс ModelVisualizer для Colab

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from IPython.display import display, Image
from pathlib import Path

# Импорт модели и датасета
from train_segmentation_advanced import (
    AttentionUNet,
    AdvancedSegmentationDataset,
    get_validation_augmentation
)

print("✅ Библиотеки для визуализации загружены")

## 13. Подготовка для визуализации

# 🎨 ДЕТАЛЬНАЯ ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ

Теперь создадим подробные визуализации для анализа качества модели!